In [1]:
# (一) OpenCV 传统图像处理（满分35分，基础必做）
# 给定一张工业零件图片part.png，使用OpenCV Python完成完整流水线处理：
# 1. 读取图像，转为灰度图；（4分）
# 2. 使用合适滤波去除图片椒盐噪声；（3分）
# 3. 形态学操作：去除微小噪点，填充零件内部细小孔洞；（7分）
# 4. 查找所有外部轮廓，筛选有效零件轮廓；（8分）
# 5. 在原图绘制筛选后的轮廓，并标注每个轮廓外接矩形、中心坐标；（8分）
# 6. 保存最终处理完成的可视化结果 `result_part.png`。（5分）

In [2]:
# 1. 读取图像，转为灰度图；（4分）
import cv2 as cv

img = cv.imread('./data/part.png')
img = cv.resize(img, (800, 500))
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

cv.imshow('gray', gray)
cv.waitKey(0)
cv.destroyAllWindows()


In [3]:
# 2. 使用合适滤波去除图片椒盐噪声；（3分）
# 中值滤波对椒盐噪声效果最佳
kernel_size = 5
median_filtered_image = cv.medianBlur(gray, kernel_size)

cv.imshow('median_filtered_image', median_filtered_image)
cv.waitKey(0)
cv.destroyAllWindows()


In [4]:
# 3. 形态学操作：去除微小噪点，填充零件内部细小孔洞；（7分）
# 自适应阈值二值化（反相）
binary = cv.adaptiveThreshold(
    median_filtered_image,      # 输入图像（灰度图）
    255,                        # 阈值最大值，前景像素设为此值
    cv.ADAPTIVE_THRESH_GAUSSIAN_C,  # 自适应方法：高斯加权平均
    cv.THRESH_BINARY_INV,       # 反相二值化：暗区域为前景(白色)
    blockSize=31,               # 邻域大小(必须是奇数)
    C=10                        # 常数，从局部均值中减去
)

# 闭运算：填充孔洞（先膨胀后腐蚀）
kernel_close = cv.getStructuringElement(cv.MORPH_RECT, (9, 9))
closed = cv.morphologyEx(binary, cv.MORPH_CLOSE, kernel_close)

# 开运算：去除噪点（先腐蚀后膨胀）
kernel_open = cv.getStructuringElement(cv.MORPH_RECT, (3, 3))
opened = cv.morphologyEx(closed, cv.MORPH_OPEN, kernel_open)

cv.imshow('closed and opened', cv.hconcat([closed, opened]))
cv.waitKey(0)
cv.destroyAllWindows()


In [5]:
# 4. 查找所有外部轮廓，筛选有效零件轮廓；（8分）
# 查找最外层轮廓（检测最外层轮廓，压缩轮廓点）
contours, _ = cv.findContours(opened, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

# 按面积筛选：保留面积大于图像1.2%的轮廓
valid_contours = []
img_area = img.shape[0] * img.shape[1]
min_area_threshold = img_area * 0.012

for contour in contours:
    if cv.contourArea(contour) >= min_area_threshold:
        valid_contours.append(contour)

print(f"检测到 {len(contours)} 个轮廓，筛选后保留 {len(valid_contours)} 个有效零件轮廓")


检测到 396 个轮廓，筛选后保留 3 个有效零件轮廓


In [6]:
# 5. 在原图绘制筛选后的轮廓，并标注每个轮廓外接矩形、中心坐标；（8分）
result = img.copy()

for i, contour in enumerate(valid_contours):
    cv.drawContours(result, [contour], -1, (0, 255, 0), 2)  # 绿色轮廓
    x, y, w, h = cv.boundingRect(contour)
    cv.rectangle(result, (x, y), (x + w, y + h), (0, 0, 255), 2)  # 红色矩形

    cx, cy = x + w // 2, y + h // 2
    cv.drawMarker(result, (cx, cy), (255, 0, 0), cv.MARKER_CROSS, 10, 2)  # 蓝色中心

    label = f"Part{i+1}({x},{y})"
    cv.putText(result, label, (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

cv.imshow('result', result)
cv.waitKey(0)
cv.destroyAllWindows()


In [7]:
# 6. 保存最终处理完成的可视化结果 `result_part.png`。（5分）
cv.imwrite('result_part.png', result)
print("处理完成，结果已保存为 result_part.png")


处理完成，结果已保存为 result_part.png
